# Experiment: Does a Linear Map from Whisper's Decoder Space to SONAR's Embedding Space Actually Hold?

**Purpose.** Before investing in the training-free contextual-biasing pipeline (trie + shallow-fusion logit
bias guided by SONAR embeddings), we must validate its single riskiest assumption:

> *A fixed matrix `W`, fitted once by least squares, can translate Whisper decoder hidden states into
> SONAR's 1024-d sentence-embedding space well enough that cosine similarity in the projected space is
> meaningful.*

If this fails, the whole "Approach 2 / Section 3" design from the guide collapses and you should switch to
a small trained adapter (Approach 1) — better to learn that now with a 30-minute experiment than after
building the full inference pipeline.

---

## Hypotheses

- **H1 (alignment holds):** projecting a *held-out* sentence's Whisper state through `W` lands near that
  sentence's true SONAR embedding — nearest-neighbor retrieval succeeds far above chance.
- **H0 (no linear alignment):** projected vectors retrieve their own sentence at ~chance level, no better
  than a control matrix fitted on deliberately shuffled (mismatched) pairs.

## Metrics & decision criteria (test pool = 400 held-out sentences, chance top-1 = 0.25%)

| Verdict | Top-1 retrieval | Top-5 | Paired cosine vs. random cosine | Action |
|---|---|---|---|---|
| **Strong — invest** | ≥ 30% | ≥ 60% | paired ≥ 0.5, random ≈ 0.0–0.1 | Build the LogitsProcessor pipeline |
| **Moderate — signal exists** | 5–30% | 20–60% | clear gap but paired < 0.5 | More calibration data, better layer, or tiny trained projector |
| **Fail — H0** | < 2% (≈ shuffled control) | < 10% | paired ≈ random | Linear map doesn't hold; use Approach 1 (trained cross-attention/adapter) |

## What this notebook does, step by step

1. **Data** — sample ~2,000 clean English sentences (WikiText-2; swap in your domain sentences later).
2. **Whisper states** — frozen `whisper-base`, teacher-forced on a *silence* spectrogram (matching the
   guide's recipe), mean-pooled over real text-token positions, from **every decoder layer** (layer sweep).
3. **SONAR embeddings** — frozen `text_sonar_basic_encoder` (auto-fallback to LaBSE if fairseq2 won't
   install, clearly flagged — that still validates the methodology, just not SONAR specifically).
4. **Fit** — OLS (`lstsq`) and Ridge (α chosen on an inner validation split), on the **train split only**.
5. **Evaluate** — top-1/5/10 retrieval, paired vs. random cosine, mean rank, linear CKA — on the
   **held-out test split**.
6. **Controls** — shuffled-pairs control fit (defines the floor), chance level.
7. **Runtime-mismatch probe** — evaluate the map on *prefix* states (25/50/75/100% of each sentence) to
   simulate the partial hypotheses Whisper actually has mid-beam-search.
8. **Verdict + saved artifacts** (`W_best.pt`, results CSV).

**Runtime:** ~5–15 min on CPU for `whisper-base` (the audio encoder is run **once** and cached — a big
speed-up over the guide's script), faster on CUDA/MPS.


---
## Section 0.1 — Dependencies

Run the `%pip` lines once, then **restart the kernel**.

Line-by-line:
- `torch transformers` — Whisper model + tokenizer/feature-extractor.
- `datasets` — pulls WikiText-2 (a few MB) for calibration sentences.
- `scikit-learn scipy` — Ridge regression and linear algebra utilities.
- `pandas matplotlib` — results tables and plots.
- `sonar-space` — Meta's SONAR pipeline. It depends on **fairseq2**, which ships wheels for Linux and
  recent macOS arm64. If this install fails on your machine, skip it — Section 3 automatically falls back
  to LaBSE and tells you so.
- `sentence-transformers` — only needed for the LaBSE fallback.


In [ ]:
# Run once, then restart the kernel.
# %pip install -U torch transformers datasets scikit-learn scipy pandas matplotlib
# %pip install sonar-space            # SONAR (primary). Needs fairseq2 — see note above.
# %pip install sentence-transformers  # fallback embedding model only


## Section 0.2 — Imports, seeds, device

Line-by-line:
- `random.seed / np.random.seed / torch.manual_seed` — every stochastic step (data shuffling, model init
  of nothing here, but also any sampling) is pinned so the experiment is exactly reproducible.
- Device selection prefers CUDA, then Apple-Silicon MPS, then CPU. Everything below is written to work on
  all three.


In [ ]:
import math, random, re, json
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("Using device:", DEVICE)


## Section 0.3 — Experiment configuration

All knobs in one dict so reruns are easy to compare.

- `whisper_model` — `whisper-base` (d_model = 512) keeps the experiment fast. If it passes here it will
  usually pass at least as well for `-small`/`-large-v3` (bigger models have richer text geometry).
- `n_sentences` — 2,000 gives 1,600 train / 400 test. The guide recommends 1,000–5,000; raise this to
  5,000 for the final fit if the experiment passes.
- `test_frac` — the held-out 20% is **never** seen by the regression. All verdict metrics come from it.
- `min_len`/`max_len` — character bounds that keep sentences "utterance-like" (roughly 8–40 spoken words).
- `ridge_alphas` — the regularization grid; the best α is chosen on an inner validation split, *not* on
  the test set.
- `prefix_fracs` / `prefix_probe_n` — settings for the runtime-mismatch probe (Section 7).


In [ ]:
CONFIG = {
    "whisper_model": "openai/whisper-base",
    "n_sentences": 2000,
    "test_frac": 0.20,
    "batch_size": 16,
    "min_len": 40,
    "max_len": 200,
    "ridge_alphas": [1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0],
    "prefix_fracs": [0.25, 0.50, 0.75, 1.00],
    "prefix_probe_n": 200,
}
CONFIG


---
## Section 1 — Calibration sentences

We need N *parallel* text samples: each sentence goes through **both** models, giving one
(Whisper-vector, SONAR-vector) pair. WikiText-2 is small, clean, and instantly downloadable.

> **For your research:** once the experiment passes on generic text, rerun with sentences from **your
> target domain** (medical, call-center, etc.) — the map you deploy should be fitted on domain text.
> Just replace the `sents` list in this cell.

Line-by-line:
1. `load_dataset("wikitext", "wikitext-2-raw-v1", split="train")` — ~36k raw lines of Wikipedia text.
2. The loop skips headings (`= Title =`) and short fragments, then splits paragraphs into sentences on
   punctuation followed by a space (`(?<=[.!?]) +` is a lookbehind so the punctuation is kept).
3. WikiText escapes punctuation as ` @-@ `, ` @,@ `, ` @.@ ` — we restore the real characters, otherwise
   Whisper's tokenizer sees garbage tokens.
4. Length filter keeps utterance-sized sentences.
5. `dict.fromkeys` deduplicates while preserving order (a `set` would destroy reproducibility).
6. Shuffle with the pinned seed, keep the first `n_sentences`.


In [ ]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

sents = []
for line in ds["text"]:
    line = line.strip()
    if not line or line.startswith("="):
        continue
    line = (line.replace(" @-@ ", "-")
                .replace(" @,@ ", ",")
                .replace(" @.@ ", "."))
    for s in re.split(r"(?<=[.!?]) +", line):
        s = s.strip()
        if CONFIG["min_len"] <= len(s) <= CONFIG["max_len"]:
            sents.append(s)

sents = list(dict.fromkeys(sents))
random.Random(SEED).shuffle(sents)
sents = sents[:CONFIG["n_sentences"]]

print(f"Collected {len(sents)} sentences.")
for s in sents[:3]:
    print(" •", s)


---
## Section 2 — Extracting Whisper decoder states (frozen)

### The recipe (and its one honest weakness)
Whisper's decoder is a *conditional* language model — its states normally depend on audio through
cross-attention. To get a text-only representation from a frozen model, we follow the guide's trick:
**teacher-force the text while feeding a silence spectrogram**. This is exactly the condition we're
testing, so the weakness is *part of the experiment*: if states extracted this way still align linearly
with SONAR, the assumption survives its own worst case.

### Speed trick the guide missed
The audio encoder's output for silence is **identical for every sentence**, so we compute it **once**,
cache it, and pass it to every decoder call via `encoder_outputs=`. This skips ~90% of the compute.

Line-by-line (this cell):
1. `WhisperProcessor` bundles the feature extractor (audio → log-mel) and the tokenizer.
2. `.eval()` disables dropout — we want deterministic states; `torch.no_grad()` later disables autograd.
3. `set_prefix_tokens(language="english", task="transcribe")` pins the special prefix to
   `<|startoftranscript|><|en|><|transcribe|><|notimestamps|>` — the same 4 tokens present at real
   inference time. `N_PREFIX` records how many there are so we can exclude them from pooling.
4. `np.zeros(16000*30)` — 30 s of digital silence at 16 kHz (Whisper always consumes 30 s windows).
   Note: we run silence through the **feature extractor**, we do *not* feed a zeros tensor as the
   spectrogram — a zero log-mel matrix is not what silence looks like, this fixes a subtle bug in the
   guide's script.
5. `whisper.get_encoder()(sil_feats)` — one encoder forward pass; `SIL_ENC` (`[1, 1500, 512]`) is reused
   for all 2,000 sentences.


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(CONFIG["whisper_model"])
whisper = (WhisperForConditionalGeneration
           .from_pretrained(CONFIG["whisper_model"])
           .to(DEVICE)
           .eval())

tok = processor.tokenizer
tok.set_prefix_tokens(language="english", task="transcribe")
N_PREFIX = len(tok.prefix_tokens)
print("Decoder prefix tokens:", tok.convert_ids_to_tokens(tok.prefix_tokens))

silence = np.zeros(16000 * 30, dtype=np.float32)
sil_feats = processor.feature_extractor(
    silence, sampling_rate=16000, return_tensors="pt"
).input_features.to(DEVICE)

with torch.no_grad():
    SIL_ENC = whisper.get_encoder()(sil_feats).last_hidden_state
print("Cached silence encoder states:", tuple(SIL_ENC.shape))


### The extraction function

Line-by-line:
1. `tok(batch, padding=True)` — tokenizes each sentence as
   `[4 prefix specials] + [text BPE tokens] + [<|endoftext|>]`, right-padding shorter sentences
   (Whisper's pad token *is* `<|endoftext|>`; `attention_mask` distinguishes real tokens from padding).
2. `SIL_ENC.expand(B, -1, -1)` — broadcast the cached encoder states across the batch **without copying
   memory**.
3. `whisper(encoder_outputs=(enc,), decoder_input_ids=ids, output_hidden_states=True)` — teacher-forced
   decoder pass. `output_hidden_states=True` returns a tuple of `n_layers+1` tensors (token embeddings +
   each transformer layer), each `[B, T, 512]`. We keep **all** of them — which layer aligns best with
   SONAR is an empirical question (mid layers often carry more "semantic" content than the final layer,
   which is specialized for next-token logits).
4. **Pooling mask:** start from `attention_mask`, zero the 4 prefix positions, and zero each sequence's
   final real token (the `<|endoftext|>`). What remains marks *actual text tokens only*. Pooling specials
   would inject constant vectors that blur every sentence together.
5. Masked mean-pool over time, then `F.normalize` (L2) — cosine geometry from here on.
6. Everything is moved to CPU float32 and concatenated into `WHISPER_STATES[layer, sentence, dim]`.


In [ ]:
@torch.no_grad()
def whisper_text_states(texts, batch_size):
    """Return pooled decoder states for every layer: tensor [n_layers+1, N, d_model]."""
    chunks = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True)
        ids  = enc.input_ids.to(DEVICE)         # [B, T]
        attn = enc.attention_mask.to(DEVICE)    # [B, T] 1 = real token, 0 = pad

        enc_states = SIL_ENC.expand(len(batch), -1, -1)
        out = whisper(encoder_outputs=(enc_states,),
                      decoder_input_ids=ids,
                      output_hidden_states=True,
                      return_dict=True)
        hs = torch.stack(out.decoder_hidden_states)   # [L+1, B, T, D]

        mask = attn.clone()
        mask[:, :N_PREFIX] = 0                              # drop the 4 special prefix tokens
        last = attn.sum(dim=1, keepdim=True) - 1            # index of <|endoftext|>
        mask.scatter_(1, last, 0)                           # drop it too
        mask = mask.unsqueeze(0).unsqueeze(-1).float()      # [1, B, T, 1] broadcast over layers/dims

        pooled = (hs * mask).sum(dim=2) / mask.sum(dim=2).clamp(min=1.0)   # [L+1, B, D]
        pooled = F.normalize(pooled, dim=-1)
        chunks.append(pooled.float().cpu())

        if (i // batch_size) % 20 == 0:
            print(f"  batch {i // batch_size + 1}/{math.ceil(len(texts) / batch_size)}")
    return torch.cat(chunks, dim=1)                          # [L+1, N, D]

WHISPER_STATES = whisper_text_states(sents, CONFIG["batch_size"])
N_LAYERS = WHISPER_STATES.shape[0]
print("WHISPER_STATES:", tuple(WHISPER_STATES.shape),
      f"→ {N_LAYERS} layers (embeddings + {N_LAYERS-1} blocks)")


---
## Section 3 — SONAR embeddings (frozen target space)

Line-by-line:
1. We *try* to build Meta's official `TextToEmbeddingModelPipeline` with the
   `text_sonar_basic_encoder` checkpoint (1024-d output). SONAR runs on CPU here for maximum
   compatibility (fairseq2's MPS support is patchy); it only runs once, so this is cheap.
2. `predict(texts, source_lang="eng_Latn")` — SONAR is multilingual and needs the language tag.
3. **Fallback:** if `sonar-space`/fairseq2 isn't installed, we drop to LaBSE (768-d). The banner makes
   the substitution impossible to miss. All downstream code is dimension-agnostic, so the *methodology*
   is still validated — but the final verdict about SONAR specifically requires the real model.
4. `F.normalize` — unit-length targets, so `X @ Y.T` is cosine similarity later.


In [ ]:
EMB_BACKEND = None
try:
    from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline
    t2vec = TextToEmbeddingModelPipeline(
        encoder="text_sonar_basic_encoder",
        tokenizer="text_sonar_basic_encoder",
        device=torch.device("cpu"),
    )
    with torch.no_grad():
        Y_raw = t2vec.predict(sents, source_lang="eng_Latn", batch_size=64).float()
    EMB_BACKEND = "SONAR (text_sonar_basic_encoder)"
except Exception as e:
    print("!" * 70)
    print("SONAR unavailable → falling back to LaBSE. Reason:", repr(e))
    print("The methodology test still runs, but rerun with real SONAR for the")
    print("final verdict before building the biasing pipeline.")
    print("!" * 70)
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("sentence-transformers/LaBSE", device=DEVICE)
    Y_raw = torch.tensor(st.encode(sents, batch_size=64, show_progress_bar=True)).float()
    EMB_BACKEND = "LaBSE fallback"

Y_ALL = F.normalize(Y_raw, dim=-1).cpu()
print(f"Target space: {EMB_BACKEND}, shape {tuple(Y_ALL.shape)}")


---
## Section 4 — Train / validation / test split

Line-by-line:
1. Shuffle sentence indices with the pinned seed.
2. Last 20% → **test** (used *only* in Section 6's evaluation — the verdict).
3. Of the remaining 80%, the last 10% → **validation** (used *only* to pick the Ridge α).
4. The rest → **train** (the regression sees only these pairs).

This three-way split is what separates a real experiment from the guide's script: without held-out
evaluation, least squares on 1024 outputs will *always* look like it "worked" on its own training data.


In [ ]:
idx = list(range(len(sents)))
random.Random(SEED).shuffle(idx)

n_test = int(len(idx) * CONFIG["test_frac"])
test_idx  = idx[-n_test:]
rest      = idx[:-n_test]
n_val     = max(1, int(len(rest) * 0.10))
val_idx   = rest[-n_val:]
train_idx = rest[:-n_val]

print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")
print(f"chance top-1 on the test pool = {100.0 / len(test_idx):.3f}%")


---
## Section 5 — Fitting and evaluation machinery

### `fit_ols(X, Y)`
`np.linalg.lstsq` solves `min_W ||X W − Y||²` exactly — the guide's `LinearRegression(fit_intercept=False)`
in one transparent line. **No intercept anywhere**: the deployed operation must be a bare matrix multiply
inside the logits processor, so the experiment must match.

### `fit_ridge(...)`
Same problem plus an L2 penalty `α‖W‖²`. With d=512 inputs and only ~1,400 training rows per output
dimension, OLS can overfit; Ridge is the honest comparison. α is chosen by **validation top-1 retrieval**
(the metric we actually care about), never by test performance.

### `evaluate_map(W, X, Y)` — the core metric
1. Project: `P = normalize(X @ W)`.
2. `sim = P @ Y.T` — cosine similarity of every projected query against every true target in the pool.
3. `rank[i]` = number of *wrong* targets that beat the true one. `rank < 1` ⇔ top-1 hit.
4. Also reports the mean **paired** cosine (diagonal) vs. mean **random** cosine (off-diagonal): a real
   alignment shows a wide gap; a fake one shows none.

### `linear_cka(X, Y)`
Centered Kernel Alignment — a fit-free score in [0, 1] of how similar the two spaces' geometries are
*before* any regression. High CKA with low retrieval suggests a fitting problem; low CKA says the spaces
just aren't linearly related.


In [ ]:
def fit_ols(X, Y):
    W, *_ = np.linalg.lstsq(X, Y, rcond=None)
    return W                                            # [d_whisper, d_target]

def fit_ridge(Xtr, Ytr, Xval, Yval, alphas):
    from sklearn.linear_model import Ridge
    best = (None, None, -1.0)                           # (W, alpha, val_top1)
    for a in alphas:
        reg = Ridge(alpha=a, fit_intercept=False).fit(Xtr, Ytr)
        W = reg.coef_.T
        val_top1 = evaluate_map(W, Xval, Yval)["top1"]
        if val_top1 > best[2]:
            best = (W, a, val_top1)
    return best

def evaluate_map(W, X, Y):
    P = F.normalize(torch.from_numpy(X).float() @ torch.from_numpy(W).float(), dim=-1)
    T = F.normalize(torch.from_numpy(Y).float(), dim=-1)
    sim = P @ T.T                                       # [N, N] cosine matrix
    d = sim.diag()
    n = sim.shape[0]
    rank = (sim > d.unsqueeze(1)).sum(dim=1)            # wrong targets beating the true one
    off_mean = (sim.sum() - d.sum()) / (n * n - n)
    return {
        "top1":  (rank < 1).float().mean().item(),
        "top5":  (rank < 5).float().mean().item(),
        "top10": (rank < 10).float().mean().item(),
        "mean_rank": rank.float().mean().item() + 1.0,
        "paired_cos": d.mean().item(),
        "random_cos": off_mean.item(),
    }

def linear_cka(X, Y):
    Xc = X - X.mean(axis=0, keepdims=True)
    Yc = Y - Y.mean(axis=0, keepdims=True)
    num = np.linalg.norm(Xc.T @ Yc, "fro") ** 2
    den = np.linalg.norm(Xc.T @ Xc, "fro") * np.linalg.norm(Yc.T @ Yc, "fro")
    return float(num / den)

print("Fitting/eval machinery defined.")


---
## Section 6 — Main result: layer sweep with OLS and Ridge

For every decoder layer ℓ (embeddings = layer 0, then blocks 1…6 for `whisper-base`):

1. Slice that layer's pooled states into train/val/test matrices (`Y` slices are shared).
2. Fit **OLS** on train; fit **Ridge** on train with α chosen on val.
3. Evaluate both on the untouched **test** pool; also record raw-space **CKA** on test.

The result table is the experiment's centerpiece — read the `top1` column against the criteria at the top.


In [ ]:
Y_np = Y_ALL.numpy()
Ytr, Yval, Yte = Y_np[train_idx], Y_np[val_idx], Y_np[test_idx]

rows, fitted = [], {}
for layer in range(N_LAYERS):
    X_np = WHISPER_STATES[layer].numpy()
    Xtr, Xval, Xte = X_np[train_idx], X_np[val_idx], X_np[test_idx]

    W_ols = fit_ols(Xtr, Ytr)
    m_ols = evaluate_map(W_ols, Xte, Yte)

    W_rdg, alpha, _ = fit_ridge(Xtr, Ytr, Xval, Yval, CONFIG["ridge_alphas"])
    m_rdg = evaluate_map(W_rdg, Xte, Yte)

    cka = linear_cka(Xte, Yte)
    fitted[layer] = {"ols": W_ols, "ridge": W_rdg, "alpha": alpha}
    for name, m in [("OLS", m_ols), (f"Ridge(α={alpha:g})", m_rdg)]:
        rows.append({"layer": layer, "method": name, "cka_raw": round(cka, 3),
                     **{k: round(v, 4) for k, v in m.items()}})
    print(f"layer {layer}: OLS top1={m_ols['top1']:.1%}  "
          f"Ridge top1={m_rdg['top1']:.1%} (α={alpha:g})  CKA={cka:.3f}")

results = pd.DataFrame(rows)
results.sort_values("top1", ascending=False).head(10)


### Shuffled-pairs control — the floor

We refit Ridge at the best layer, but with `Y` rows **randomly permuted**, destroying the true
text↔embedding correspondence while keeping every marginal statistic identical. Whatever score this
control achieves is what "the regression found nothing real" looks like. Your genuine result must beat
it decisively; if it doesn't, any apparent alignment was an artifact.


In [ ]:
best_row = results.loc[results["top1"].idxmax()]
BEST_LAYER = int(best_row["layer"])
W_BEST = fitted[BEST_LAYER]["ridge"] if "Ridge" in best_row["method"] else fitted[BEST_LAYER]["ols"]
print(f"Best config: layer {BEST_LAYER}, {best_row['method']}, test top-1 = {best_row['top1']:.1%}")

X_np = WHISPER_STATES[BEST_LAYER].numpy()
Xtr, Xval, Xte = X_np[train_idx], X_np[val_idx], X_np[test_idx]

perm = np.random.RandomState(SEED).permutation(len(train_idx))
W_ctrl, a_ctrl, _ = fit_ridge(Xtr, Ytr[perm], Xval, Yval, CONFIG["ridge_alphas"])
m_ctrl = evaluate_map(W_ctrl, Xte, Yte)

print(f"Shuffled control: top1={m_ctrl['top1']:.2%}  top5={m_ctrl['top5']:.2%}  "
      f"paired_cos={m_ctrl['paired_cos']:.3f}  random_cos={m_ctrl['random_cos']:.3f}")
print(f"Chance level:     top1={1/len(test_idx):.2%}")


### Visual summary

- **Left:** test top-1 retrieval per decoder layer (OLS vs Ridge), with the chance and shuffled-control
  floors. Expect an inverted-U: early layers are too lexical, the last layer is specialized for logits.
- **Right:** at the best configuration, the distribution of *paired* cosines (projected vector vs. its own
  SONAR embedding) against *random-pair* cosines. A healthy alignment separates these two histograms.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for method, grp in results.groupby(results["method"].str.startswith("Ridge")):
    label = "Ridge" if method else "OLS"
    g = grp.sort_values("layer")
    axes[0].plot(g["layer"], g["top1"] * 100, marker="o", label=label)
axes[0].axhline(100 / len(test_idx), ls=":", c="gray", label="chance")
axes[0].axhline(m_ctrl["top1"] * 100, ls="--", c="red", label="shuffled control")
axes[0].set_xlabel("decoder layer"); axes[0].set_ylabel("test top-1 retrieval (%)")
axes[0].set_title("Linear-map quality by layer"); axes[0].legend()

P = F.normalize(torch.from_numpy(Xte).float() @ torch.from_numpy(W_BEST).float(), dim=-1)
T = F.normalize(torch.from_numpy(Yte).float(), dim=-1)
sim = P @ T.T
paired = sim.diag().numpy()
off = sim[~torch.eye(len(Xte), dtype=bool)].numpy()
axes[1].hist(off, bins=60, alpha=0.6, density=True, label="random pairs")
axes[1].hist(paired, bins=60, alpha=0.6, density=True, label="true pairs")
axes[1].set_xlabel("cosine similarity"); axes[1].set_ylabel("density")
axes[1].set_title(f"Best config (layer {BEST_LAYER}) — target: {EMB_BACKEND}")
axes[1].legend()
plt.tight_layout(); plt.show()


---
## Section 7 — Runtime-mismatch probe: does the map survive *partial* sentences?

The deployed biasing system will apply `W` to decoder states **mid-generation**, when Whisper has only
emitted part of a sentence — but we fitted `W` on states pooled over *complete* sentences. This probe
measures how much that distribution shift costs.

Because the decoder is **causal** (position *t* attends only to positions ≤ *t*), the hidden states of a
prefix inside a full teacher-forced pass are *bit-identical* to running the prefix alone — so one forward
pass per sentence gives us every prefix for free.

Line-by-line:
1. `token_states_one` — single-sentence forward at `BEST_LAYER`; slice off the 4 prefix specials and the
   trailing `<|endoftext|>`, leaving `[n_text_tokens, 512]`.
2. For each fraction *f* in {25%, 50%, 75%, 100%}: pool the first `ceil(f·n)` token states, normalize,
   project through `W_BEST`, renormalize.
3. Retrieval target: the sentence's **full** SONAR embedding, searched among all 400 test embeddings —
   i.e. "given a half-finished hypothesis, can we still tell *which* context sentence it matches?", which
   is precisely the question the trie-biasing scorer will ask.

Interpretation: a graceful decline (e.g. 100% → 50% prefix losing only a modest chunk of top-5) is fine —
during decoding you'd accumulate evidence over steps. A cliff at any partial fraction means sentence-level
fitting doesn't transfer and you should refit `W` on prefix-pooled states (an easy variant of Section 2).


In [ ]:
@torch.no_grad()
def token_states_one(text, layer):
    ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
    out = whisper(encoder_outputs=(SIL_ENC,), decoder_input_ids=ids,
                  output_hidden_states=True, return_dict=True)
    hs = out.decoder_hidden_states[layer][0]            # [T, D]
    return hs[N_PREFIX:-1].float().cpu()                # text tokens only

n_probe = min(CONFIG["prefix_probe_n"], len(test_idx))
sub_sents = [sents[i] for i in test_idx[:n_probe]]
token_cache = [token_states_one(s, BEST_LAYER) for s in sub_sents]

W_t = torch.from_numpy(W_BEST).float()
Y_pool = F.normalize(torch.from_numpy(Yte).float(), dim=-1)   # all 400 test targets
own = torch.arange(n_probe)                                   # row i's true target is column i

probe_rows = []
for frac in CONFIG["prefix_fracs"]:
    preds = []
    for ts in token_cache:
        k = max(1, math.ceil(frac * ts.shape[0]))
        preds.append(F.normalize(ts[:k].mean(dim=0, keepdim=True), dim=-1))
    P = F.normalize(torch.cat(preds) @ W_t, dim=-1)
    sim = P @ Y_pool.T
    d = sim[own, own]
    rank = (sim > d.unsqueeze(1)).sum(dim=1)
    probe_rows.append({"prefix": f"{int(frac*100)}%",
                       "top1": (rank < 1).float().mean().item(),
                       "top5": (rank < 5).float().mean().item(),
                       "paired_cos": d.mean().item()})
prefix_df = pd.DataFrame(probe_rows)

ax = prefix_df.set_index("prefix")[["top1", "top5"]].plot.bar(figsize=(7, 4), rot=0)
ax.axhline(1 / len(test_idx), ls=":", c="gray")
ax.set_ylabel("retrieval accuracy"); ax.set_title("Does the map survive partial sentences?")
plt.tight_layout(); plt.show()
prefix_df


---
## Section 8 — Save artifacts

- `whisper_to_sonar_W.pt` — the winning matrix (a plain `[512, 1024]` tensor: at runtime,
  `state @ W` then cosine against hotword embeddings). Named honestly — no PCA is involved,
  unlike the guide's `_pca.pt`.
- `alignment_results.csv` — the full sweep table for your records.
- `alignment_meta.json` — config + best layer + control numbers, so the artifact is self-describing.


In [ ]:
torch.save(torch.from_numpy(W_BEST).float(), "whisper_to_sonar_W.pt")
results.to_csv("alignment_results.csv", index=False)
meta = {
    "config": CONFIG, "embedding_backend": EMB_BACKEND,
    "best_layer": BEST_LAYER, "best_method": str(best_row["method"]),
    "test_top1": float(best_row["top1"]), "test_top5": float(best_row["top5"]),
    "shuffled_control_top1": m_ctrl["top1"],
    "chance_top1": 1 / len(test_idx),
    "prefix_probe": probe_rows,
}
with open("alignment_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2)[:600], "...")


---
## Section 9 — Reading the verdict

Compare your numbers against the criteria table at the top. In brief:

- **Strong (top-1 ≥ 30%, wide cosine gap, control ≈ chance, graceful prefix decline)** → the linear
  bridge is real. Next steps: (1) refit on 5,000 *domain* sentences, (2) refit at the best layer using
  prefix-pooled states so training matches runtime, (3) build the `LogitsProcessor` that walks the trie
  and adds `λ · cos(state @ W, hotword_emb)` to candidate-token logits, (4) grid-search λ on a dev set
  with real audio.

- **Moderate (5–30%)** → geometry is partially shared. Cheapest upgrades, in order: more calibration
  sentences; try `whisper-small` (richer 768-d states); fit on prefix-pooled states; replace the linear
  map with a 2-layer MLP projector trained in minutes on the same frozen pairs — still no ASR fine-tuning,
  slightly outside "pure linear" but far cheaper than Approach 1.

- **Fail (≈ shuffled control)** → silence-conditioned Whisper states do not encode linearly-recoverable
  SONAR semantics. Two escalation paths: extract states from **real audio** (e.g. LibriSpeech clips +
  their transcripts — states will match deployment much better), or accept that a trained cross-attention
  adapter (the guide's Approach 1) is required.

### Known limitations of this experiment (by design)
1. States come from silence-conditioned teacher forcing — deliberately matching the guide's recipe; the
   real-audio variant above is the natural follow-up.
2. Sentence-level retrieval is a *proxy* for token-level biasing utility; passing here is necessary, not
   sufficient. The end-to-end test is WER/hotword-recall on real audio with the full pipeline.
3. If you ran the LaBSE fallback, rerun with real SONAR before deciding anything about SONAR.
